## 0. Conexión

In [55]:
from pathlib import Path
import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "analytics_engineer_assets").exists())
DB = ROOT / "warehouse.duckdb"

con = duckdb.connect(str(DB), read_only=True)

def q(sql):
    """Ejecuta una consulta y muestra el resultado como tabla."""
    con.sql(sql).show(max_rows=50)

q("SELECT table_schema, table_name FROM information_schema.tables ORDER BY 1, 2")

┌──────────────┬──────────────────────────┐
│ table_schema │        table_name        │
│   varchar    │         varchar          │
├──────────────┼──────────────────────────┤
│ intermediate │ int_fx_rates_to_usd      │
│ intermediate │ int_order_items_enriched │
│ intermediate │ int_orders_enriched      │
│ marts        │ dim_currencies           │
│ marts        │ dim_customers            │
│ marts        │ dim_date                 │
│ marts        │ dim_hour                 │
│ marts        │ dim_products             │
│ marts        │ fct_fx_rates             │
│ marts        │ fct_order_items          │
│ marts        │ fct_orders               │
│ raw          │ customers                │
│ raw          │ fx_rates                 │
│ raw          │ order_items              │
│ raw          │ orders                   │
│ raw          │ products                 │
│ staging      │ stg_customers            │
│ staging      │ stg_fx_rates             │
│ staging      │ stg_order_items

## 1. Vista general de cada tabla

In [56]:
q("SELECT * FROM raw.orders LIMIT 10")

┌─────────┬─────────────┬─────────────────────┬──────────────┬──────────┬───────────┬───────────────────────────────┬──────────────┐
│   id    │ customer_id │     order_date      │ total_amount │ currency │  status   │          _loaded_at           │ _source_file │
│ varchar │   varchar   │       varchar       │   varchar    │ varchar  │  varchar  │   timestamp with time zone    │   varchar    │
├─────────┼─────────────┼─────────────────────┼──────────────┼──────────┼───────────┼───────────────────────────────┼──────────────┤
│ 1       │ 1           │ 2024-01-16 14:30:00 │ 89.99        │ USD      │ completed │ 2026-09-18 22:06:02.517799+02 │ orders.csv   │
│ 2       │ 2           │ 2024-01-21 10:15:00 │ 156.5        │ GBP      │ completed │ 2026-09-18 22:06:02.517799+02 │ orders.csv   │
│ 3       │ 3           │ 2024-01-26 16:45:00 │ 234.75       │ EUR      │ completed │ 2026-09-18 22:06:02.517799+02 │ orders.csv   │
│ 4       │ 4           │ 2024-01-28 11:20:00 │ 67.25        │ EUR   

In [57]:
q("SELECT * FROM raw.order_items LIMIT 10")

┌─────────┬──────────┬────────────┬──────────┬────────────┬──────────┬───────────────────────────────┬─────────────────┐
│   id    │ order_id │ product_id │ quantity │ unit_price │ currency │          _loaded_at           │  _source_file   │
│ varchar │ varchar  │  varchar   │ varchar  │  varchar   │ varchar  │   timestamp with time zone    │     varchar     │
├─────────┼──────────┼────────────┼──────────┼────────────┼──────────┼───────────────────────────────┼─────────────────┤
│ 1       │ 1        │ 1          │ 1        │ 89.99      │ USD      │ 2026-09-18 22:06:02.603737+02 │ order_items.csv │
│ 2       │ 2        │ 15         │ 2        │ 78.25      │ GBP      │ 2026-09-18 22:06:02.603737+02 │ order_items.csv │
│ 3       │ 3        │ 23         │ 1        │ 234.75     │ EUR      │ 2026-09-18 22:06:02.603737+02 │ order_items.csv │
│ 4       │ 4        │ 45         │ 3        │ 22.42      │ EUR      │ 2026-09-18 22:06:02.603737+02 │ order_items.csv │
│ 5       │ 5        │ 8        

In [58]:
q("SELECT id, name, category, base_price, currency FROM raw.products LIMIT 10")

┌───────┬───────────────────────────────┬─────────────┬────────────┬──────────┐
│  id   │             name              │  category   │ base_price │ currency │
│ int64 │            varchar            │   varchar   │   double   │ varchar  │
├───────┼───────────────────────────────┼─────────────┼────────────┼──────────┤
│     1 │ Wireless Bluetooth Headphones │ Electronics │      89.99 │ USD      │
│     2 │ Smart Fitness Watch           │ Electronics │     299.99 │ USD      │
│     3 │ Portable Bluetooth Speaker    │ Electronics │     356.25 │ USD      │
│     4 │ 4K Webcam                     │ Electronics │     367.25 │ USD      │
│     5 │ Wireless Charging Pad         │ Electronics │     423.75 │ USD      │
│     6 │ Gaming Mechanical Keyboard    │ Electronics │      345.6 │ USD      │
│     7 │ Ergonomic Wireless Mouse      │ Electronics │     189.99 │ USD      │
│     8 │ Tablet Stand Adjustable       │ Electronics │     299.99 │ USD      │
│     9 │ USB-C Hub 7-in-1              

In [59]:
q("SELECT * FROM raw.customers LIMIT 10")

┌─────────┬───────────────┬─────────────────────────┬─────────────────────┬─────────┬───────────────────────────────┬───────────────┐
│   id    │     name      │          email          │  registration_date  │ country │          _loaded_at           │ _source_file  │
│ varchar │    varchar    │         varchar         │       varchar       │ varchar │   timestamp with time zone    │    varchar    │
├─────────┼───────────────┼─────────────────────────┼─────────────────────┼─────────┼───────────────────────────────┼───────────────┤
│ 1       │ John Smith    │ john.smith@email.com    │ 2024-01-15 10:30:00 │ USA     │ 2026-09-18 22:06:02.466849+02 │ customers.csv │
│ 2       │ Emma Johnson  │ emma.johnson@email.com  │ 2024-01-20 14:15:00 │ UK      │ 2026-09-18 22:06:02.466849+02 │ customers.csv │
│ 3       │ Pierre Dubois │ pierre.dubois@email.com │ 2024-01-25 09:45:00 │ France  │ 2026-09-18 22:06:02.466849+02 │ customers.csv │
│ 4       │ Maria Garcia  │ maria.garcia@email.com  │ 2024-02-

In [60]:
q("SELECT * FROM raw.fx_rates")

┌───────────────┬────────────┬───────────────────────────────────────┬───────────────────────────────┬───────────────┐
│ base_currency │ rate_date  │                 rates                 │          _loaded_at           │ _source_file  │
│    varchar    │  varchar   │                 json                  │   timestamp with time zone    │    varchar    │
├───────────────┼────────────┼───────────────────────────────────────┼───────────────────────────────┼───────────────┤
│ USD           │ 2024-06-01 │ {"USD":1.0,"EUR":0.9285,"GBP":0.7911} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ EUR           │ 2024-06-01 │ {"USD":1.077,"EUR":1.0,"GBP":0.852}   │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ GBP           │ 2024-06-01 │ {"USD":1.2641,"EUR":1.1737,"GBP":1.0} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ USD           │ 2024-09-15 │ {"USD":1.0,"EUR":0.9042,"GBP":0.7628} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ EUR           │ 2024-09-15 │ {"USD":1.1059,"EU

### Volumen y rango de fechas

In [61]:
q("""
    SELECT 'customers' AS tabla, count(*) AS filas, count(DISTINCT id) AS ids_distintos FROM raw.customers
    UNION ALL SELECT 'orders',      count(*), count(DISTINCT id) FROM raw.orders
    UNION ALL SELECT 'order_items', count(*), count(DISTINCT id) FROM raw.order_items
    UNION ALL SELECT 'products',    count(*), count(DISTINCT id) FROM raw.products
""")

q("""
    SELECT min(order_date::TIMESTAMP) AS primera_orden,
           max(order_date::TIMESTAMP) AS ultima_orden
    FROM raw.orders
""")

┌─────────────┬───────┬───────────────┐
│    tabla    │ filas │ ids_distintos │
│   varchar   │ int64 │     int64     │
├─────────────┼───────┼───────────────┤
│ customers   │    50 │            50 │
│ orders      │   453 │           453 │
│ order_items │   363 │           363 │
│ products    │   100 │           100 │
└─────────────┴───────┴───────────────┘

┌─────────────────────┬─────────────────────┐
│    primera_orden    │    ultima_orden     │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2024-01-16 14:30:00 │ 2024-12-31 15:55:00 │
└─────────────────────┴─────────────────────┘



### Status de las órdenes

Todas están `completed`. Igual se testea con `accepted_values` para detectar valores nuevos en el futuro.

In [1]:
q("SELECT status, count(*) AS ordenes FROM raw.orders GROUP BY status")

NameError: name 'q' is not defined

monedas inválidas

`XYZ`, `ABC` y `QWE` no son códigos ISO 4217. Aparecen tanto en órdenes como en ítems.

**Decisión:** se conservan las filas, con `revenue_usd` nulo y un flag de moneda inválida. Cuentan para unidades vendidas, pero no para revenue.

In [ ]:
q("""
    SELECT currency, count(*) AS ordenes,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM raw.orders
    GROUP BY currency
    ORDER BY ordenes DESC
""")

┌──────────┬─────────┬────────┐
│ currency │ ordenes │  pct   │
│ varchar  │  int64  │ double │
├──────────┼─────────┼────────┤
│ USD      │     244 │   53.9 │
│ EUR      │     126 │   27.8 │
│ XYZ      │      27 │    6.0 │
│ ABC      │      27 │    6.0 │
│ QWE      │      25 │    5.5 │
│ GBP      │       4 │    0.9 │
└──────────┴─────────┴────────┘



In [ ]:
q("""
    SELECT currency, count(*) AS items,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM raw.order_items
    GROUP BY currency
    ORDER BY items DESC
""")

┌──────────┬───────┬────────┐
│ currency │ items │  pct   │
│ varchar  │ int64 │ double │
├──────────┼───────┼────────┤
│ USD      │   186 │   51.2 │
│ EUR      │    99 │   27.3 │
│ ABC      │    26 │    7.2 │
│ XYZ      │    25 │    6.9 │
│ QWE      │    24 │    6.6 │
│ GBP      │     3 │    0.8 │
└──────────┴───────┴────────┘



productos fuera del catálogo

`order_items.product_id` no es una FK garantizada. Los IDs 101 a 110 no existen en `products.json`, y todas esas líneas son de agosto en adelante.

**Decisión:** no se descartan. Se crea un miembro "producto desconocido" en `dim_products` por cada ID huérfano, con `is_in_catalogue = false`.

In [ ]:
q("""
    SELECT oi.product_id, count(*) AS lineas
    FROM raw.order_items oi
    LEFT JOIN raw.products p ON p.id = oi.product_id::INT
    WHERE p.id IS NULL
    GROUP BY oi.product_id
    ORDER BY oi.product_id::INT
""")

┌────────────┬────────┐
│ product_id │ lineas │
│  varchar   │ int64  │
├────────────┼────────┤
│ 101        │      9 │
│ 102        │      8 │
│ 103        │      7 │
│ 104        │      8 │
│ 105        │      7 │
│ 106        │      8 │
│ 107        │      7 │
│ 108        │      7 │
│ 109        │      6 │
│ 110        │      4 │
└────────────┴────────┘
  10 rows   2 columns



In [ ]:
q("""
    SELECT CASE WHEN p.id IS NULL THEN 'fuera de catálogo' ELSE 'en catálogo' END AS estado,
           count(*) AS lineas,
           min(o.order_date::TIMESTAMP) AS desde,
           max(o.order_date::TIMESTAMP) AS hasta
    FROM raw.order_items oi
    JOIN raw.orders o ON o.id = oi.order_id
    LEFT JOIN raw.products p ON p.id = oi.product_id::INT
    GROUP BY estado
""")

┌───────────────────┬────────┬─────────────────────┬─────────────────────┐
│      estado       │ lineas │        desde        │        hasta        │
│      varchar      │ int64  │      timestamp      │      timestamp      │
├───────────────────┼────────┼─────────────────────┼─────────────────────┤
│ fuera de catálogo │     71 │ 2024-08-01 10:15:00 │ 2024-12-30 09:55:00 │
│ en catálogo       │    292 │ 2024-01-16 14:30:00 │ 2024-12-31 13:15:00 │
└───────────────────┴────────┴─────────────────────┴─────────────────────┘



órdenes sin líneas de ítems

Las órdenes con ID 361 a 453 no tienen ninguna línea en `order_items`.

 quedan fuera del análisis por producto (no sabemos qué se vendió), pero sí cuentan para el análisis por hora del día.

In [ ]:
q("""
    SELECT count(*) AS ordenes_sin_items,
           min(o.id::INT) AS id_desde,
           max(o.id::INT) AS id_hasta
    FROM raw.orders o
    WHERE NOT EXISTS (SELECT 1 FROM raw.order_items oi WHERE oi.order_id = o.id)
""")

┌───────────────────┬──────────┬──────────┐
│ ordenes_sin_items │ id_desde │ id_hasta │
│       int64       │  int32   │  int32   │
├───────────────────┼──────────┼──────────┤
│                93 │      361 │      453 │
└───────────────────┴──────────┴──────────┘



 moneda de la orden distinta a la del ítem

 el revenue de cada línea se calcula con la moneda de la línea.

In [ ]:
q("""
    SELECT o.currency AS moneda_orden, oi.currency AS moneda_item, count(*) AS lineas
    FROM raw.orders o
    JOIN raw.order_items oi ON oi.order_id = o.id
    WHERE o.currency <> oi.currency
    GROUP BY ALL
    ORDER BY lineas DESC
""")

┌──────────────┬─────────────┬────────┐
│ moneda_orden │ moneda_item │ lineas │
│   varchar    │   varchar   │ int64  │
├──────────────┼─────────────┼────────┤
│ USD          │ EUR         │     18 │
│ EUR          │ USD         │     15 │
│ USD          │ XYZ         │     13 │
│ USD          │ QWE         │     10 │
│ EUR          │ ABC         │     10 │
│ XYZ          │ USD         │      7 │
│ USD          │ ABC         │      7 │
│ ABC          │ USD         │      6 │
│ QWE          │ USD         │      5 │
│ EUR          │ XYZ         │      5 │
│ QWE          │ EUR         │      4 │
│ XYZ          │ EUR         │      3 │
│ XYZ          │ QWE         │      2 │
│ ABC          │ EUR         │      2 │
│ EUR          │ QWE         │      2 │
│ ABC          │ QWE         │      2 │
│ XYZ          │ ABC         │      1 │
│ GBP          │ USD         │      1 │
│ ABC          │ XYZ         │      1 │
└──────────────┴─────────────┴────────┘
  19 rows                   3 columns



total de la orden vs. suma de sus líneas

Muchas diferencias son de centavos (redondeo de `quantity × unit_price`), pero otras son grandes.

 el revenue por producto sale de las líneas. Un test con tolerancia distingue el redondeo de las diferencias reales.

In [ ]:
q("""
    WITH comparacion AS (
        SELECT o.id,
               o.total_amount::DOUBLE AS total_orden,
               sum(oi.quantity::INT * oi.unit_price::DOUBLE) AS suma_lineas
        FROM raw.orders o
        JOIN raw.order_items oi ON oi.order_id = o.id
        GROUP BY o.id, o.total_amount
    )
    SELECT CASE
             WHEN abs(total_orden - suma_lineas) <= 0.05 THEN '1) coincide o redondeo (<= 0.05)'
             WHEN abs(total_orden - suma_lineas) <= 1    THEN '2) diferencia chica (<= 1)'
             ELSE                                             '3) diferencia real (> 1)'
           END AS categoria,
           count(*) AS ordenes
    FROM comparacion
    GROUP BY categoria
    ORDER BY categoria
""")

┌──────────────────────────────────┬─────────┐
│            categoria             │ ordenes │
│             varchar              │  int64  │
├──────────────────────────────────┼─────────┤
│ 1) coincide o redondeo (<= 0.05) │     198 │
│ 2) diferencia chica (<= 1)       │      12 │
│ 3) diferencia real (> 1)         │     150 │
└──────────────────────────────────┴─────────┘



In [ ]:
q("""
    SELECT o.id,
           o.total_amount::DOUBLE AS total_orden,
           round(sum(oi.quantity::INT * oi.unit_price::DOUBLE), 2) AS suma_lineas
    FROM raw.orders o
    JOIN raw.order_items oi ON oi.order_id = o.id
    GROUP BY o.id, o.total_amount
    HAVING abs(total_orden - suma_lineas) > 1
    ORDER BY o.id::INT
    LIMIT 10
""")

┌─────────┬─────────────┬─────────────┐
│   id    │ total_orden │ suma_lineas │
│ varchar │   double    │   double    │
├─────────┼─────────────┼─────────────┤
│ 150     │       178.7 │      177.03 │
│ 202     │      345.65 │      156.81 │
│ 203     │       234.7 │      156.51 │
│ 204     │      298.45 │      234.85 │
│ 205     │      167.95 │      188.49 │
│ 206     │      445.85 │      422.52 │
│ 208     │       356.9 │      423.96 │
│ 209     │       123.5 │       167.5 │
│ 210     │      278.85 │      423.66 │
│ 211     │       234.6 │      423.65 │
└─────────┴─────────────┴─────────────┘
  10 rows                   3 columns



 precio unitario vs. precio de catálogo

En la mayoría de las líneas `unit_price` es igual al `base_price` del catálogo (en USD), **aunque la línea diga EUR o GBP**. Eso sugiere que la etiqueta de moneda en origen podría no ser confiable.
 no se corrige (no podemos saberlo con certeza), pero se documenta como riesgo.

In [ ]:
q("""
    SELECT oi.currency AS moneda_linea,
           count(*) AS lineas_en_catalogo,
           sum(CASE WHEN oi.unit_price::DOUBLE = p.base_price THEN 1 ELSE 0 END) AS precio_igual_catalogo
    FROM raw.order_items oi
    JOIN raw.products p ON p.id = oi.product_id::INT
    GROUP BY oi.currency
    ORDER BY lineas_en_catalogo DESC
""")

┌──────────────┬────────────────────┬───────────────────────┐
│ moneda_linea │ lineas_en_catalogo │ precio_igual_catalogo │
│   varchar    │       int64        │        int128         │
├──────────────┼────────────────────┼───────────────────────┤
│ USD          │                184 │                    45 │
│ EUR          │                 98 │                    25 │
│ GBP          │                  3 │                     1 │
│ QWE          │                  3 │                     0 │
│ ABC          │                  2 │                     0 │
│ XYZ          │                  2 │                     0 │
└──────────────┴────────────────────┴───────────────────────┘



Tasas de cambio

`fx_rates.json` solo trae dos fechas (2024-06-01 y 2024-09-15), pero hay órdenes desde enero. Además, GBP como moneda base solo aparece en junio.

- Cada orden toma la última tasa anterior o igual a su fecha (join por rango de vigencia o `ASOF JOIN`).
- Las órdenes previas a 2024-06-01 usan la primera tasa disponible y quedan marcadas con un flag.
- Si falta el par directo (ej. GBP→USD en septiembre), se usa la inversa de USD→GBP.

In [72]:
q("""
    SELECT base_currency,
           rate_date::DATE AS rate_date,
           k AS moneda_destino,
           (rates ->> k)::DOUBLE AS tasa
    FROM raw.fx_rates, unnest(json_keys(rates)) AS t(k)
    ORDER BY rate_date, base_currency, moneda_destino
""")

┌───────────────┬────────────┬────────────────┬────────┐
│ base_currency │ rate_date  │ moneda_destino │  tasa  │
│    varchar    │    date    │    varchar     │ double │
├───────────────┼────────────┼────────────────┼────────┤
│ EUR           │ 2024-06-01 │ EUR            │    1.0 │
│ EUR           │ 2024-06-01 │ GBP            │  0.852 │
│ EUR           │ 2024-06-01 │ USD            │  1.077 │
│ GBP           │ 2024-06-01 │ EUR            │ 1.1737 │
│ GBP           │ 2024-06-01 │ GBP            │    1.0 │
│ GBP           │ 2024-06-01 │ USD            │ 1.2641 │
│ USD           │ 2024-06-01 │ EUR            │ 0.9285 │
│ USD           │ 2024-06-01 │ GBP            │ 0.7911 │
│ USD           │ 2024-06-01 │ USD            │    1.0 │
│ EUR           │ 2024-09-15 │ EUR            │    1.0 │
│ EUR           │ 2024-09-15 │ GBP            │ 0.8436 │
│ EUR           │ 2024-09-15 │ USD            │ 1.1059 │
│ USD           │ 2024-09-15 │ EUR            │ 0.9042 │
│ USD           │ 2024-09-15 │ 

In [73]:
q("""
    SELECT CASE WHEN order_date::DATE < DATE '2024-06-01' THEN 'antes de la primera tasa'
                ELSE 'con tasa disponible' END AS cobertura,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY cobertura
""")

┌──────────────────────────┬─────────┐
│        cobertura         │ ordenes │
│         varchar          │  int64  │
├──────────────────────────┼─────────┤
│ con tasa disponible      │     398 │
│ antes de la primera tasa │      55 │
└──────────────────────────┴─────────┘



Patrones temporales (anticipo de la pregunta de negocio 2)

- Solo hay órdenes entre las 9 y las 18 h: parece horario local de negocio. La zona horaria no está documentada y es un supuesto a declarar.
- El volumen salta en agosto (de ~13 a ~77 órdenes por mes). Hay que tenerlo en cuenta al interpretar promedios.

In [74]:
q("""
    SELECT hour(order_date::TIMESTAMP) AS hora, count(*) AS ordenes
    FROM raw.orders
    GROUP BY hora
    ORDER BY hora
""")

┌───────┬─────────┐
│ hora  │ ordenes │
│ int64 │  int64  │
├───────┼─────────┤
│     9 │      37 │
│    10 │      57 │
│    11 │      56 │
│    12 │      55 │
│    13 │      36 │
│    14 │      64 │
│    15 │      72 │
│    16 │      41 │
│    17 │      18 │
│    18 │      17 │
└───────┴─────────┘
      10 rows    



In [75]:
q("""
    SELECT dayname(order_date::TIMESTAMP) AS dia,
           isodow(order_date::TIMESTAMP)  AS n_dia,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY ALL
    ORDER BY n_dia
""")

┌───────────┬───────┬─────────┐
│    dia    │ n_dia │ ordenes │
│  varchar  │ int64 │  int64  │
├───────────┼───────┼─────────┤
│ Monday    │     1 │      64 │
│ Tuesday   │     2 │      67 │
│ Wednesday │     3 │      61 │
│ Thursday  │     4 │      65 │
│ Friday    │     5 │      66 │
│ Saturday  │     6 │      64 │
│ Sunday    │     7 │      66 │
└───────────┴───────┴─────────┘



In [76]:
q("""
    SELECT strftime(date_trunc('month', order_date::TIMESTAMP), '%Y-%m') AS mes,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY mes
    ORDER BY mes
""")

┌─────────┬─────────┐
│   mes   │ ordenes │
│ varchar │  int64  │
├─────────┼─────────┤
│ 2024-01 │       5 │
│ 2024-02 │      10 │
│ 2024-03 │      15 │
│ 2024-04 │      15 │
│ 2024-05 │      10 │
│ 2024-06 │      15 │
│ 2024-07 │      13 │
│ 2024-08 │      78 │
│ 2024-09 │      75 │
│ 2024-10 │      77 │
│ 2024-11 │      75 │
│ 2024-12 │      65 │
└─────────┴─────────┘
       12 rows     



 Integridad referencial restante

In [77]:
q("""
    SELECT 'orders con customer inexistente' AS chequeo, count(*) AS filas
    FROM raw.orders o
    WHERE NOT EXISTS (SELECT 1 FROM raw.customers c WHERE c.id = o.customer_id)
    UNION ALL
    SELECT 'order_items con order inexistente', count(*)
    FROM raw.order_items oi
    WHERE NOT EXISTS (SELECT 1 FROM raw.orders o WHERE o.id = oi.order_id)
    UNION ALL
    SELECT 'order_items con quantity <= 0', count(*)
    FROM raw.order_items WHERE quantity::INT <= 0
    UNION ALL
    SELECT 'order_items con unit_price <= 0', count(*)
    FROM raw.order_items WHERE unit_price::DOUBLE <= 0
""")

┌───────────────────────────────────┬───────┐
│              chequeo              │ filas │
│              varchar              │ int64 │
├───────────────────────────────────┼───────┤
│ orders con customer inexistente   │     0 │
│ order_items con order inexistente │     0 │
│ order_items con quantity <= 0     │     0 │
│ order_items con unit_price <= 0   │     0 │
└───────────────────────────────────┴───────┘



 Resumen de calidad de datos

Tabla consolidada para copiar en `design_notes.md`.

In [78]:
q("""
    WITH valid AS (SELECT unnest(['USD', 'EUR', 'GBP']) AS c),
    line_totals AS (
        SELECT order_id, sum(quantity::INT * unit_price::DOUBLE) AS s
        FROM raw.order_items GROUP BY order_id
    )
    SELECT 'Órdenes con moneda inválida' AS problema,
           count(*) FILTER (WHERE currency NOT IN (SELECT c FROM valid)) AS afectadas,
           count(*) AS total
    FROM raw.orders
    UNION ALL
    SELECT 'Ítems con moneda inválida',
           count(*) FILTER (WHERE currency NOT IN (SELECT c FROM valid)), count(*)
    FROM raw.order_items
    UNION ALL
    SELECT 'Ítems con producto fuera de catálogo',
           count(*) FILTER (WHERE product_id::INT NOT IN (SELECT id FROM raw.products)), count(*)
    FROM raw.order_items
    UNION ALL
    SELECT 'Órdenes sin ítems',
           count(*) FILTER (WHERE id NOT IN (SELECT order_id FROM raw.order_items)), count(*)
    FROM raw.orders
    UNION ALL
    SELECT 'Ítems con moneda distinta a su orden',
           count(*) FILTER (WHERE oi.currency <> o.currency), count(*)
    FROM raw.order_items oi JOIN raw.orders o ON o.id = oi.order_id
    UNION ALL
    SELECT 'Órdenes cuyo total difiere de sus líneas (> 1)',
           count(*) FILTER (WHERE abs(o.total_amount::DOUBLE - lt.s) > 1), count(*)
    FROM raw.orders o JOIN line_totals lt ON lt.order_id = o.id
    UNION ALL
    SELECT 'Órdenes anteriores a la primera tasa FX',
           count(*) FILTER (WHERE order_date::DATE < DATE '2024-06-01'), count(*)
    FROM raw.orders
""")

┌────────────────────────────────────────────────┬───────────┬───────┐
│                    problema                    │ afectadas │ total │
│                    varchar                     │   int64   │ int64 │
├────────────────────────────────────────────────┼───────────┼───────┤
│ Órdenes con moneda inválida                    │        79 │   453 │
│ Ítems con moneda inválida                      │        75 │   363 │
│ Ítems con producto fuera de catálogo           │        71 │   363 │
│ Órdenes sin ítems                              │        93 │   453 │
│ Ítems con moneda distinta a su orden           │       114 │   363 │
│ Órdenes cuyo total difiere de sus líneas (> 1) │       150 │   360 │
│ Órdenes anteriores a la primera tasa FX        │        55 │   453 │
└────────────────────────────────────────────────┴───────────┴───────┘



In [79]:
q("SELECT * FROM intermediate.int_fx_rates_to_usd ORDER BY currency_code, rate_date")


┌───────────────────┬───────────────┬────────────┬───────────────┬────────────────┬────────────┬────────────┐
│ fx_rate_to_usd_id │ currency_code │ rate_date  │  rate_to_usd  │ fx_rate_source │ valid_from │  valid_to  │
│      varchar      │    varchar    │    date    │ decimal(18,6) │    varchar     │    date    │    date    │
├───────────────────┼───────────────┼────────────┼───────────────┼────────────────┼────────────┼────────────┤
│ EUR_2024-06-01    │ EUR           │ 2024-06-01 │      1.077000 │ direct         │ 1900-01-01 │ 2024-09-15 │
│ EUR_2024-09-15    │ EUR           │ 2024-09-15 │      1.105900 │ direct         │ 2024-09-15 │ 9999-12-31 │
│ GBP_2024-06-01    │ GBP           │ 2024-06-01 │      1.264100 │ direct         │ 1900-01-01 │ 2024-09-15 │
│ GBP_2024-09-15    │ GBP           │ 2024-09-15 │      1.310960 │ inverse        │ 2024-09-15 │ 9999-12-31 │
│ USD_2024-06-01    │ USD           │ 2024-06-01 │      1.000000 │ identity       │ 1900-01-01 │ 2024-09-15 │
│ USD_2024

In [80]:
q("SELECT order_item_id, currency_code, order_date, fx_rate_to_usd, line_amount, line_amount_usd, is_valid_currency, is_in_catalogue FROM intermediate.int_order_items_enriched LIMIT 20")

┌───────────────┬───────────────┬────────────┬────────────────┬───────────────┬─────────────────┬───────────────────┬─────────────────┐
│ order_item_id │ currency_code │ order_date │ fx_rate_to_usd │  line_amount  │ line_amount_usd │ is_valid_currency │ is_in_catalogue │
│     int32     │    varchar    │    date    │ decimal(18,6)  │ decimal(18,2) │  decimal(18,2)  │      boolean      │     boolean     │
├───────────────┼───────────────┼────────────┼────────────────┼───────────────┼─────────────────┼───────────────────┼─────────────────┤
│           110 │ EUR           │ 2024-09-16 │       1.105900 │        356.40 │          394.14 │ true              │ true            │
│           111 │ USD           │ 2024-09-18 │       1.000000 │        123.76 │          123.76 │ true              │ true            │
│           112 │ USD           │ 2024-09-20 │       1.000000 │        298.55 │          298.55 │ true              │ true            │
│           113 │ EUR           │ 2024-09-22 │  

In [81]:
q("SELECT p.product_name, p.category, sum(f.quantity) AS unidades, sum(f.line_amount_usd) AS revenue_usd FROM marts.fct_order_items f JOIN marts.dim_products p USING (product_id) GROUP BY ALL ORDER BY unidades DESC LIMIT 10;")

┌──────────────────────────┬──────────────┬──────────┬───────────────┐
│       product_name       │   category   │ unidades │  revenue_usd  │
│         varchar          │   varchar    │  int128  │ decimal(38,2) │
├──────────────────────────┼──────────────┼──────────┼───────────────┤
│ Dog Training Treats      │ Pet Supplies │       40 │       1027.43 │
│ Emergency Car Kit        │ Automotive   │       28 │        721.38 │
│ Dog Leash Retractable    │ Pet Supplies │       27 │        783.00 │
│ Cat Litter Mat           │ Pet Supplies │       27 │        436.32 │
│ Pet Carrier Soft         │ Pet Supplies │       26 │        525.14 │
│ Dashboard Camera HD      │ Automotive   │       24 │       1270.12 │
│ Unknown product (id 104) │ Unknown      │       24 │          NULL │
│ Puzzle 1000 Pieces       │ Toys & Games │       24 │        722.59 │
│ Dog Toy Rope             │ Pet Supplies │       24 │        699.10 │
│ Cat Scratching Post      │ Pet Supplies │       21 │        247.24 │
└─────

In [82]:
con.close()
print("Conexión cerrada")

Conexión cerrada
